# 1단계 PLATOON1C — 판정과 확인

`PLATOON1C_t09`, `PLATOON1C_t20`, `PLATOON1C_t30`의 252판이 모두 끝난 뒤 **런타임 하나**(CPU + 높은 RAM)에서 **모두 실행**합니다.
1. 판정(몇 분, SUMO 없음): 문턱마다 **학습의 몫 L = 후보 − TM4**가 0.3초 이상·하한 > 0·양수 날짜 5개 이상이고 ARENA3 판정을 통과하는지 봅니다.
   고정신호·P0와 비교한 표도 함께 나옵니다.
2. 확인(약 4~5시간): 통과한 문턱 중 가장 좋은 하나만 아직 쓰지 않은 확인 84상황에서 잽니다. 통과가 없으면 SUMO를 돌리지 않고 끝납니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_PLATOON1C.zip"
PLATOON0 = DRIVE / "explore_platoon0"          # 무리 도착 세계 M4·TM4·OM4·P1M4(읽기만)
PLATOON1 = DRIVE / "explore_platoon1"          # 학생 3개와 PT1 평가(읽기만)
PBASE = DRIVE / "explore_platoon_base"         # 무리 도착 세계 고정신호·P0(읽기만)
STUDENTS = PLATOON1 / "students"
OUT = DRIVE / "explore_platoon1c"
WORKSPACE = Path("/content/FutureTraffic_PLATOON1C")
MIN_CPUS = 8
for path in (ARCHIVE, PLATOON0 / "experiment.json", PLATOON0 / "runs" / "P" / "TM4", PLATOON1 / "eval" / "PT1" / "experiment.json",
             PBASE / "runs" / "P" / "FIXC", *(STUDENTS / f"seed_{s}" / "model.pt" for s in (42, 43, 44))):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", OUT)
if CPUS < MIN_CPUS:
    print(f"주의: CPU가 {CPUS}개입니다. 확인을 돌리게 되면 오래 걸립니다. 가능하면 CPU + 높은 RAM 런타임을 쓰세요.")
counts = {c: len(list((OUT / "eval" / c / "runs").glob(f"{c}_m*/*/cache.json"))) for c in ("t09", "t20", "t30")}
assert set(counts.values()) == {252}, f"아직 끝나지 않은 평가가 있습니다: {counts}"

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "PLATOON1C_V1", "PLATOON1C 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("platoon1c_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON1C 설정 {package_manifest['platoon1c_config_sha256'][:12]} | 사전 등록 {package_manifest['prereg_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

for name in ("test_explore_platoon1c.py", "test_explore_platoon1.py"):
    colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", name], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 판정(SUMO 없음, 몇 분). 숫자는 맥에서 다시 계산합니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
code = colab.run_live(["src/explore_platoon1c.py", "judge", "--eval", str(OUT / "eval"), "--platoon0", str(PLATOON0),
                       "--platoon1", str(PLATOON1), "--pbase", str(PBASE), "--out", str(OUT / "judge")], WORKSPACE, check=False)
assert code == 0, "판정에서 오류가 났습니다. 위 출력을 저에게 보내 주세요."
display(Markdown((OUT / "judge" / "summary.md").read_text()))

In [ ]:
# 2) 확인(확인 84상황, 무리 도착 세계): 1)이 고른 문턱이 있을 때만 M4·TM4·후보를 돌립니다(약 4~5시간, 백그라운드).
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
LOGS = OUT / "logs"
pids = colab.launch({"confirm_PT1C": ["src/explore_platoon1c.py", "confirm", "--judge", str(OUT / "judge"), "--students", str(STUDENTS),
                                      "--out", str(OUT / "judge" / "confirm"), "--work", "/content/pc_confirm", "--workers", str(CPUS)]},
                    WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 600)
result = colab.read_json(OUT / "judge" / "confirm" / "confirmation.json")
print("확인 결과:", result and result.get("status"))
if (OUT / "judge" / "confirm" / "summary.md").exists():
    display(Markdown((OUT / "judge" / "confirm" / "summary.md").read_text()))